In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "cross-session_7-channels-feet-rest"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "TTK", "Zhou2016", "BNCI2014_001",  "Zhou2020" ]


In [4]:
for pipeline in [*mi_pipelines.values(), *ood_pipelines.values()]:
    channel_params = {key: tuple(COMMON_CHANNELS) for key in pipeline.get_params(deep=True) if key.split("__")[-1] == "channel_names"}
    pipeline.set_params(**channel_params)


MI_METHODS = ['ACM+TS+SVM', 'FBCSP+ElasticNet', 'ShallowConvNet', 'EEGNet', "Microstates+GB", "VAR+shLDA", "Log-bandpower+shLDA" ]
OOD_METHODS = ['ACM+TS+OCSVM', 'FBCSP+IsolationForest', 'ShallowConvNet+kNN', 'EEGNet+TempDens', "Log-bandpower+Mahalanobis", "Microstates+LOF", "VAR+Mahalanobis" ]


In [5]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None

        exp_type = "mi"
        for method, pipeline in mi_pipelines.items():
            if method not in MI_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": exp_type, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, duration=3.0)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        #data = None
        for method, pipeline in ood_pipelines.items():
            if method not in OOD_METHODS:
                continue
            exp_ood = [ ("ood-feet", ("left_foot", "right_foot") if dataset == "TTK" else ("feet",)),
                        ("ood-rest", ("rest",)) ]
            for experiment_type, ood in exp_ood:
                result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": experiment_type, "method": method}
                if result_path(result).exists():
                    continue
                #if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, duration=3.0, ood_classes=ood )
                train_data, val_data, test_data = data

                model = train(train_data, pipeline)

                result["OOD_AUROC"] = OOD_AUROC(test_data, model)
                save_result(result)
                print(f"{dataset} s{subject} {method} ({ood}): {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood-feet", metric="OOD_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood-rest", metric="OOD_AUROC")

TTK


,MI_BA,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.629,6,0.860,0.640,0.500,0.550,0.725,0.500
FBCSP+ElasticNet,0.624,6,0.860,0.550,0.510,0.540,0.746,0.540
EEGNet,0.599,6,0.700,0.590,0.520,0.550,0.716,0.520
ShallowConvNet,0.596,6,0.720,0.580,0.490,0.510,0.677,0.600
Log-bandpower+shLDA,0.593,6,0.690,0.550,0.550,0.580,0.649,0.540
VAR+shLDA,0.556,6,0.620,0.620,0.510,0.530,0.595,0.460
Microstates+GB,0.533,6,0.590,0.510,0.450,0.530,0.628,0.490


,MI_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.725,6,0.943,0.661,0.642,0.672,0.821,0.610
FBCSP+ElasticNet,0.675,6,0.945,0.538,0.616,0.525,0.816,0.608
EEGNet,0.654,6,0.815,0.651,0.513,0.556,0.810,0.581
ShallowConvNet,0.626,6,0.782,0.648,0.450,0.560,0.716,0.600
Log-bandpower+shLDA,0.619,6,0.720,0.630,0.542,0.515,0.709,0.600
VAR+shLDA,0.589,6,0.626,0.620,0.657,0.538,0.578,0.516
Microstates+GB,0.568,6,0.707,0.509,0.498,0.546,0.660,0.490


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
EEGNet+TempDens,0.782,6,0.915,0.644,0.650,0.796,0.922,0.764
FBCSP+IsolationForest,0.773,6,0.839,0.749,0.688,0.794,0.846,0.724
ShallowConvNet+kNN,0.758,6,0.898,0.677,0.742,0.844,0.786,0.602
Log-bandpower+Mahalanobis,0.637,6,0.683,0.754,0.567,0.711,0.581,0.524
VAR+Mahalanobis,0.631,6,0.656,0.679,0.580,0.596,0.630,0.647
ACM+TS+OCSVM,0.588,6,0.586,0.576,0.505,0.635,0.659,0.567
Microstates+LOF,0.572,6,0.709,0.521,0.547,0.678,0.530,0.447


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
FBCSP+IsolationForest,0.702,6,0.706,0.740,0.616,0.885,0.624,0.643
ShallowConvNet+kNN,0.680,6,0.711,0.757,0.741,0.920,0.476,0.474
Log-bandpower+Mahalanobis,0.630,6,0.618,0.676,0.602,0.836,0.499,0.551
EEGNet+TempDens,0.600,6,0.742,0.457,0.498,0.829,0.629,0.448
VAR+Mahalanobis,0.588,6,0.622,0.635,0.506,0.670,0.573,0.521
ACM+TS+OCSVM,0.534,6,0.591,0.421,0.511,0.720,0.430,0.530
Microstates+LOF,0.494,6,0.615,0.394,0.456,0.714,0.365,0.420


Zhou2016


... (10 outputs left out)

,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.583,20,0.533,0.483,0.625,0.583,0.492,0.783,0.875,0.567,0.525,0.558,0.583,0.533,0.600,0.567,0.683,0.550,0.500,0.600,0.542,0.483
EEGNet,0.577,20,0.483,0.533,0.642,0.483,0.550,0.725,0.917,0.475,0.550,0.517,0.567,0.575,0.542,0.617,0.775,0.475,0.550,0.525,0.542,0.500
FBCSP+ElasticNet,0.551,20,0.500,0.500,0.667,0.467,0.500,0.558,0.817,0.492,0.550,0.525,0.742,0.483,0.542,0.558,0.558,0.492,0.533,0.550,0.517,0.475
Microstates+GB,0.545,20,0.592,0.492,0.667,0.450,0.508,0.567,0.767,0.508,0.508,0.600,0.500,0.500,0.633,0.517,0.567,0.442,0.417,0.583,0.517,0.567
VAR+shLDA,0.538,20,0.492,0.500,0.533,0.550,0.525,0.658,0.667,0.442,0.475,0.517,0.525,0.492,0.550,0.583,0.608,0.525,0.517,0.525,0.567,0.517
Log-bandpower+shLDA,0.535,20,0.525,0.458,0.608,0.508,0.508,0.583,0.717,0.500,0.542,0.517,0.475,0.542,0.508,0.567,0.608,0.500,0.483,0.550,0.508,0.492
ACM+TS+SVM,0.535,20,0.500,0.467,0.650,0.467,0.500,0.500,0.708,0.483,0.550,0.500,0.542,0.542,0.517,0.600,0.525,0.508,0.500,0.617,0.517,0.500


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.607,20,0.539,0.465,0.691,0.590,0.538,0.863,0.924,0.568,0.528,0.602,0.599,0.585,0.614,0.602,0.787,0.556,0.480,0.592,0.532,0.476
ACM+TS+SVM,0.604,20,0.468,0.484,0.754,0.457,0.513,0.786,0.841,0.469,0.621,0.490,0.616,0.662,0.562,0.614,0.769,0.528,0.482,0.649,0.638,0.681
EEGNet,0.603,20,0.536,0.515,0.725,0.472,0.567,0.834,0.967,0.460,0.549,0.483,0.633,0.658,0.566,0.644,0.852,0.538,0.475,0.583,0.522,0.477
FBCSP+ElasticNet,0.582,20,0.501,0.497,0.682,0.498,0.528,0.780,0.869,0.439,0.512,0.579,0.750,0.535,0.543,0.594,0.784,0.504,0.477,0.616,0.520,0.441
Microstates+GB,0.574,20,0.673,0.459,0.721,0.389,0.495,0.612,0.881,0.514,0.523,0.633,0.553,0.530,0.666,0.545,0.642,0.450,0.465,0.572,0.535,0.617
VAR+shLDA,0.559,20,0.504,0.538,0.602,0.467,0.530,0.719,0.728,0.483,0.423,0.518,0.565,0.518,0.581,0.627,0.595,0.520,0.488,0.620,0.548,0.599
Log-bandpower+shLDA,0.559,20,0.531,0.468,0.646,0.518,0.480,0.669,0.796,0.485,0.482,0.552,0.502,0.531,0.564,0.639,0.671,0.497,0.488,0.583,0.572,0.501


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.629,20,0.507,0.520,0.579,0.639,0.500,0.885,0.747,0.485,0.606,0.491,0.670,0.617,0.626,0.754,0.926,0.712,0.482,0.674,0.452,0.704
FBCSP+IsolationForest,0.587,20,0.516,0.485,0.597,0.402,0.627,0.861,0.630,0.459,0.618,0.486,0.659,0.585,0.539,0.767,0.826,0.561,0.426,0.621,0.423,0.647
ShallowConvNet+kNN,0.572,20,0.530,0.535,0.555,0.636,0.604,0.883,0.692,0.489,0.567,0.473,0.583,0.604,0.546,0.810,0.352,0.655,0.484,0.597,0.413,0.428
Log-bandpower+Mahalanobis,0.535,20,0.536,0.449,0.581,0.564,0.545,0.392,0.525,0.473,0.705,0.519,0.528,0.583,0.495,0.639,0.582,0.509,0.468,0.579,0.490,0.530
VAR+Mahalanobis,0.531,20,0.554,0.479,0.589,0.529,0.566,0.430,0.521,0.467,0.536,0.467,0.496,0.574,0.510,0.692,0.662,0.494,0.438,0.596,0.474,0.546
Microstates+LOF,0.516,20,0.450,0.533,0.503,0.473,0.579,0.703,0.498,0.459,0.584,0.549,0.561,0.537,0.444,0.635,0.365,0.473,0.475,0.428,0.512,0.554
ACM+TS+OCSVM,0.512,20,0.474,0.444,0.556,0.532,0.545,0.664,0.548,0.463,0.499,0.457,0.477,0.499,0.523,0.546,0.486,0.552,0.448,0.489,0.460,0.580


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.635,20,0.462,0.688,0.672,0.624,0.631,0.746,0.886,0.536,0.579,0.518,0.567,0.633,0.517,0.707,0.937,0.619,0.614,0.745,0.446,0.565
FBCSP+IsolationForest,0.612,20,0.515,0.653,0.607,0.463,0.642,0.765,0.824,0.506,0.571,0.477,0.575,0.616,0.481,0.806,0.910,0.522,0.559,0.710,0.457,0.570
Log-bandpower+Mahalanobis,0.573,20,0.528,0.601,0.554,0.598,0.584,0.553,0.554,0.501,0.621,0.552,0.483,0.643,0.450,0.714,0.735,0.530,0.564,0.627,0.529,0.533
VAR+Mahalanobis,0.565,20,0.534,0.600,0.597,0.519,0.538,0.481,0.648,0.540,0.511,0.508,0.555,0.591,0.462,0.657,0.816,0.535,0.457,0.668,0.473,0.604
ShallowConvNet+kNN,0.558,20,0.532,0.690,0.580,0.644,0.555,0.762,0.797,0.465,0.526,0.451,0.581,0.602,0.434,0.780,0.299,0.465,0.458,0.704,0.360,0.477
ACM+TS+OCSVM,0.556,20,0.455,0.593,0.671,0.548,0.578,0.495,0.672,0.550,0.531,0.481,0.456,0.550,0.455,0.672,0.719,0.539,0.588,0.582,0.468,0.518
Microstates+LOF,0.533,20,0.454,0.659,0.577,0.511,0.564,0.579,0.596,0.586,0.570,0.535,0.605,0.490,0.518,0.622,0.310,0.444,0.529,0.455,0.537,0.516


In [6]:
display_summary_by_dataset(
    experiment,
    MI_METHODS,
    OOD_METHODS,
    datasets=datasets,
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD feet", "ood-feet", "OOD_AUROC"),
        ("OOD rest", "ood-rest", "OOD_AUROC"),
    ],
)